In [1]:
import numpy as np
import torch
import torch.utils.data as data
import torch.nn as nn
import sys
import warnings
from itertools import product

### NoteBook environment specific (has to be configured on user side)

In [ ]:
# import github repo into notebook file directory, might require fixing path
!rm -r /Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

# path to the github repo
root_dir = "/root/Architectural-Biases-in-Time-Series-Anomaly-Detection"
sys.path.append(root_dir)

import utils.config as config
# arg1: path to the cats dataset, arg2: path to the checkpoint folder
config.init("/mnt/data/data.csv", root_dir + "/checkpoint_dir")

rm: cannot remove '/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 414, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 414 (delta 28), reused 43 (delta 16), pack-reused 356 (from 1)
Receiving objects: 100% (414/414), 119.00 MiB | 14.73 MiB/s, done.
Resolving deltas: 100% (214/214), done.
fatal: cannot change to '/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory


### Setup the environment to have access to required repo functions

In [ ]:
warnings.filterwarnings("ignore")
from models.lstm_forecaster import lstm_forecaster
from models.lstm_ae import lstm_ae
from models.transformer_encoder_forecaster import patch_transformer
from utils.residual_scoring import score_custom_N2RE
from utils.residual_scoring import fit_custom_N2RE
import dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)

2.8.0+cu129
cuda


### Random states for reproducibility

In [5]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Initializing all datasets

In [ ]:
train_dataset_lstm_f = dataset.forecasting_Dataset(
    device, 100, 4, start = 0, end = 1000000)
val_dataset_lstm_f = dataset.forecasting_Dataset(
    device, 100, 4, scaler = train_dataset_lstm_f.scaler, 
    start = 1000000, end = 3500000, train = False)
test_dataset_lstm_f = dataset.forecasting_Dataset(
    device, 100, 4, scaler = train_dataset_lstm_f.scaler, 
    start = 3500000, end = 5000000, train = False)

train_dataset_lstm_ae = dataset.AE_Dataset(
    device, 256, start = 0, end = 1000000)
val_dataset_lstm_ae = dataset.AE_Dataset(
    device, 256, scaler = train_dataset_lstm_ae.scaler, 
    start = 1000000, end = 3500000, train = False)
test_dataset_lstm_ae = dataset.AE_Dataset(
    device, 256, scaler = train_dataset_lstm_ae.scaler, 
    start = 3500000, end = 5000000, train = False)

train_dataset_transformer_f = dataset.forecasting_Dataset(
    device, 256, 4, start = 0, end = 1000000)
val_dataset_transformer_f = dataset.forecasting_Dataset(
    device, 256, 4, scaler = train_dataset_transformer_f.scaler, 
    start = 1000000, end = 3500000, train = False)
test_dataset_transformer_f = dataset.forecasting_Dataset(
    device, 256, 4, scaler = train_dataset_transformer_f.scaler, 
    start = 3500000, end = 5000000, train = False)

## Initializing models

In [ ]:
lstm_forecaster_model = lstm_forecaster(
    hidden_size = 128, horizon = 4, num_layers = 1).to(device)
lstm_forecaster_model.load_state_dict(
    torch.load(root_dir + '/saved_model_weights/lstm_forecaster_weights.pth'))

lstm_ae_model = lstm_ae(
    17, lookback_window = 256, embedding_dim_ratio = 0.55).to(device)
lstm_ae_model.load_state_dict(
    torch.load(root_dir + '/saved_model_weights/lstm_AutoEncoder_weights.pth'))

transformer_model = patch_transformer(
    lookback_window = 256, 
    forecast_horizon = 4,
    d_model = 256, 
    nhead = 8, 
    dropout = 0.0, 
    num_features = 17, 
    num_blocks = 1
).to(device)
transformer_model.load_state_dict(
    torch.load(root_dir + '/saved_model_weights/transformer_forecaster_weights.pt'))

32


<All keys matched successfully>

## Computing scores

In [9]:
def scoring_pipeline(device, model, train_dataset, val_dataset, test_dataset):
    stats = fit_custom_N2RE(device, model, train_dataset)
    val_scores, val_labels, val_cats = score_custom_N2RE(device, model, val_dataset, *stats)
    test_scores, test_labels, test_cats = score_custom_N2RE(device, model, test_dataset, *stats)
    return {
        "val": (val_scores, val_labels, val_cats),
        "test": (test_scores, test_labels, test_cats),
    }

In [ ]:
lstm_f_results = scoring_pipeline(
    device, lstm_forecaster_model, 
    train_dataset_lstm_f, 
    val_dataset_lstm_f, 
    test_dataset_lstm_f)
lstm_ae_results = scoring_pipeline(
    device, 
    lstm_ae_model, 
    train_dataset_lstm_ae, 
    val_dataset_lstm_ae, 
    test_dataset_lstm_ae)
transformer_results = scoring_pipeline(
    device, 
    transformer_model, 
    train_dataset_transformer_f, 
    val_dataset_transformer_f, 
    test_dataset_transformer_f)

computing residuals
Beginning Inference
Beginning Inference
computing residuals
Beginning Inference
Beginning Inference
computing residuals
Beginning Inference
Beginning Inference


## Saving into dict

In [11]:
def save_results(save_path, results):
    np.savez_compressed(
        save_path,
        v_scores=results["val"][0],
        v_labels=results["val"][1],
        v_categories=results["val"][2],
        t_scores=results["test"][0],
        t_labels=results["test"][1],
        t_categories=results["test"][2],
    )

In [13]:
import os
results_dir = os.path.join(root_dir, "saved_results")
os.makedirs(results_dir, exist_ok=True)
save_results(os.path.join(results_dir, "lstm_f_results.npz"), lstm_f_results)
save_results(os.path.join(results_dir, "lstm_ae_results.npz"), lstm_ae_results)
save_results(os.path.join(results_dir, "transformer_f_results.npz"), transformer_results)